# Colab track — no Docker, no GPU, no live OLMo

This is the **demo** copy of `T1-09_prompt_vs_rag.ipynb`. Replies that would have come from OLMo/Phi are **scripted** (`[OLMo demo]` / `[Phi-4-mini demo]`). Days 2–3 already work without a model.

The live-hosting course is `tier1/` + Docker (or [03_own_environment.md](https://github.com/justkrismanohar/fine-tuning-build/blob/main/03_own_environment.md)).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justkrismanohar/fine-tuning-build/blob/main/colab/day2/T1-09_prompt_vs_rag.ipynb)

**Runtime → CPU.** Then **Runtime → Run all**. First code cell clones the course (or finds it if you already have the folder).


In [1]:
# [RUN] COLAB TRACK — find or clone the course, stay on demo replies
import os
import sys
import subprocess
from pathlib import Path

os.environ["CIVICDESK_COLAB"] = "1"
os.environ["CIVICDESK_MODE"] = "demo"

def _course_root() -> Path:
    markers = Path("tier1") / "day1" / "civicdesk_lab.py"
    for base in [Path("/content/fine-tuning-build"), Path.cwd(), *Path.cwd().resolve().parents]:
        if (base / markers).is_file():
            return base
    dest = Path("/content/fine-tuning-build")
    url = "https://github.com/justkrismanohar/fine-tuning-build.git"
    print("Cloning", url)
    subprocess.check_call(["git", "clone", "--depth", "1", url, str(dest)])
    return dest

ROOT = _course_root()
DAY = ROOT / "tier1" / "day2"
os.chdir(DAY)
if str(DAY) not in sys.path:
    sys.path.insert(0, str(DAY))
if "day2" == "day3":
    d2 = str(ROOT / "tier1" / "day2")
    if d2 not in sys.path:
        sys.path.insert(0, d2)
print("COLAB TRACK")
print(" course:", ROOT)
print(" cwd   :", Path.cwd())
print(" Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.")

# Markdown images in Colab resolve from /content, not cwd
import shutil
content = Path("/content") if Path("/content").is_dir() else Path.cwd()
for folder in ("images", "cards"):
    src = DAY / folder
    dest = content / folder
    if src.is_dir() and dest.resolve() != src.resolve():
        shutil.copytree(src, dest, dirs_exist_ok=True)
        print(f" {folder} →", dest)

# Day 2 helper lives here; Day 3 notebooks that import rag also add ../day2


COLAB TRACK
 course: G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course
 cwd   : G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2
 Live OLMo/Phi are OFF. Later chat cells use canned CivicDesk replies.


# T1-09 Prompt vs RAG

**Time:** 12:45–13:45 &nbsp;|&nbsp; **Evidence:** side-by-side log — invented circular vs cited paragraph vs abstain

**Notebook learning outcomes**

- Show the **same question** answered from weights only vs from a retrieved chunk.
- Name **hallucination** when a circular number appears with no source.
- **Abstain** when the corpus has no paragraph (Q-MISSING).


**How to use this notebook**

- **No-code / code-light:** run every cell in order. Read the teaching note after a cell. Fill the write-up. Do not paste work files.
- **Occasional-code:** run all cells. Change only lines marked `# [CHANGE]`.
- **Deep-code:** optional cells are tagged `# [OPTIONAL]`. Stay with the room on the required path. Do not drive someone else's notebook.

Git is not required. If you are stuck, load the matching file in `completed/` for this notebook only, mark it as loaded from reference, and continue.


## Overnight readers

**Prompt-only** here means: the generator never sees a handbook chunk. It continues from the standing instruction plus the user text. That is Day 1.

**RAG** means: retrieve first, then generate from visible chunks.

The failure mode ministries actually fear is not a blank. It is a **fluent wrong**: “CivicDesk Circular 99 of 2024 appears to allow…” — invented, confident, unusable in Finance or Defence.

![Same question, two paths](../../tier1/day2/images/t2_prompt_vs_rag.jpg)

**Extra words:**

- **Grounded** — every factual clause points at a chunk id.
- **Ungrounded / hallucinated** — the clause cannot be found in the prompt or the retrieved set.
- **Abstain** — the honest third output. Not a shrug: a logged stop.


## Ministry link

Finance / Defence / Health cannot accept an invented circular, OPR note, or guideline number. Labour cannot accept an invented C2. Trade cannot accept “your password has been reset” because the model sounded sure.

This lab is **N3**. The abstain on Q-MISSING is the skill to take home.


## 1. Same question, two replies


In [2]:
# [RUN]
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}["Q-ATTACH"]
print("USER:", q["user"])
print()
print("--- prompt-only (no retrieval) ---")
print(rag.answer_prompt_only(q["user"]))
print()
print("--- RAG ---")
out = rag.answer_with_rag(q["user"], role="clerk")
print(out["reply"])
print("mode:", out["mode"], "chunks:", out["visible_ids"])
rag.save_log("T1-09_attach", {"prompt_only": rag.answer_prompt_only(q["user"]), "rag": out})


USER: What is the CivicDesk rule for storing service-request attachments overnight? Cite the handbook.

--- prompt-only (no retrieval) ---
[prompt-only demo] CivicDesk Circular 99 of 2024 appears to allow overnight storage anywhere convenient. (No handbook was retrieved. This circular is invented. Fail.)

--- RAG ---
Cited CivicDesk Circular HB-ATTACH-04. Quoted span: 'Service-request attachments may be stored overnight only on the CivicDesk approved file store. They must not be copied into personal email or a public chatbot. If the store is unav…' I used only retrieved CivicDesk text. Human review if this is a consequential act.
mode: cite chunks: ['HB-ATTACH-04', 'HB-CLASS-01']
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-09_attach.json


### Teaching note

Mark the prompt-only reply: it **fails** even if it reads like a circular. Fluency is not evidence.

The RAG reply must show `HB-ATTACH-04` (or another approved chunk that actually answers). If your overlap retriever missed, that is a retrieval bug — still better than a fake circular, because you can see the empty hit list.


## 2. Missing source — abstain drill

Q-MISSING asks for CivicDesk Circular 88 of 2019 on drone permits. That document is **not** in the pile. The correct product behaviour is stop.


In [3]:
# [RUN]
import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}["Q-MISSING"]
print("USER:", q["user"])
print("expect_chunk:", q["expect_chunk"])
print()
print("hits:", [(h["id"], h["score"]) for h in rag.retrieve(q["user"], k=3)])
out = rag.answer_with_rag(q["user"], role="clerk")
print()
print(out["reply"])
print("mode:", out["mode"])
rag.save_log("T1-09_missing", out)


USER: What does CivicDesk Circular 88 of 2019 say about drone permits over the harbour?
expect_chunk: None

hits: []

I cannot cite. No approved paragraph for this role is in the retrieved set. I will not invent a circular number. Stop.
mode: abstain
saved G:\Other computers\My Laptop\Phd\UWI\19-2026-2027-SEM-1\15-UNDP Consultant\course\tier1\day2\logs\T1-09_missing.json


### After you run

`mode` should be **abstain**. Hits may be empty or weakly related. Weak overlap is not permission to invent Circular 88.

**Write-up:** one sentence you would put in a service-desk script: *If the chunk is not in the retrieved set, we will not…*


In [4]:
# [CHANGE] occasional-code: swap the question id.
QUESTION_ID = "Q-C2"  # try Q-PWD

import civicdesk_rag as rag

q = {x["id"]: x for x in rag.load_questions()}[QUESTION_ID]
print("prompt-only:")
print(rag.answer_prompt_only(q["user"]))
print()
print("RAG:")
print(rag.answer_with_rag(q["user"])["reply"])


prompt-only:
[prompt-only demo] CivicDesk Circular 99 of 2024 appears to allow overnight storage anywhere convenient. (No handbook was retrieved. This circular is invented. Fail.)

RAG:
Cited CivicDesk Vacancy Handbook v3, C2. Quoted span: 'C2. The applicant must hold a recognised diploma in records management or equivalent. Library studies is not automatically equivalent. If equivalent is claimed, mark Human review r…' I used only retrieved CivicDesk text. Human review if this is a consequential act.
